<a href="https://colab.research.google.com/github/sanaisrail/code-switching-codesaviours-si26-sana/blob/main/SI26_Week7_Sana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset (2).csv")

print(df.head())
print("\nColumns:")
print(df.columns.tolist())

print("\nShape:")
print(df.shape)

                                            sentence    word label
0  Aaj mera mood bilkul off hai, so I just want t...     Aaj   URD
1  Aaj mera mood bilkul off hai, so I just want t...    mera   URD
2  Aaj mera mood bilkul off hai, so I just want t...    mood   URD
3  Aaj mera mood bilkul off hai, so I just want t...  bilkul   URD
4  Aaj mera mood bilkul off hai, so I just want t...     off   URD

Columns:
['sentence', 'word', 'label']

Shape:
(2061, 3)


In [4]:
print("Labels:")
print(df["label"].value_counts())

print("\nUnique labels:")
print(df["label"].unique())

Labels:
label
ENG    1058
URD    1003
Name: count, dtype: int64

Unique labels:
['URD' 'ENG']


In [5]:
# Check how many sentences contain both URD and ENG words

sentence_labels = df.groupby("sentence")["label"].unique()

mixed_sentences = sentence_labels[
    sentence_labels.apply(lambda x: len(x) > 1)
]

print("Total sentences:", len(sentence_labels))
print("Mixed-language sentences:", len(mixed_sentences))

print("\nExample mixed sentences:")
for sentence in mixed_sentences.index[:5]:
    print(sentence)

Total sentences: 170
Mixed-language sentences: 170

Example mixed sentences:
Aaj bohot zyada assignments hain, I don't know how I will finish them.
Aaj class ke baad we can discuss the project, agar tum free ho.
Aaj class mein new topic start hua, and it was actually interesting.
Aaj class mein students bohot active thay, everyone participated in the discussion.
Aaj hum friends ke saath shopping karne ja rahe hain, do you want to join?


In [6]:
sentences = (
    df.groupby("sentence")
      .apply(
          lambda x: {
              "words": x["word"].tolist(),
              "labels": x["label"].tolist()
          }
      )
      .tolist()
)

print("Total grouped sentences:", len(sentences))

print("\nFirst sentence:")
print(sentences[0])

Total grouped sentences: 170

First sentence:
{'words': ['Aaj', 'bohot', 'zyada', 'assignments', 'hain,', 'I', "don't", 'know', 'how', 'I', 'will', 'finish', 'them.'], 'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']}


/tmp/ipykernel_2504/1165086530.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [7]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 136
Testing sentences: 34


In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

model_name = "xlm-roberta-base"

# Tokenizer load karo
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model load karo
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={
        0: "URD",
        1: "ENG"
    },
    label2id={
        "URD": 0,
        "ENG": 1
    }
)

print("Model loaded successfully!")
print("Model:", model_name)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Model: xlm-roberta-base


In [9]:
# Label mapping
label2id = {
    "URD": 0,
    "ENG": 1
}

id2label = {
    0: "URD",
    1: "ENG"
}


def tokenize_and_align_labels(examples):

    # Words ko tokens mein convert karo
    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    # Har sentence ke labels process karo
    for i, label in enumerate(examples["labels"]):

        # Har token kis original word se belong karta hai
        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:

            # Special token
            if word_id is None:
                label_ids.append(-100)

            # Word ka first token
            elif word_id != previous_word_id:
                label_ids.append(
                    label2id[label[word_id]]
                )

            # Same word ka additional sub-token
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized

print("Tokenization function ready!")

Tokenization function ready!


In [10]:
from datasets import Dataset

def to_hf_dataset(data):
    return Dataset.from_dict({
        "words": [item["words"] for item in data],
        "labels": [item["labels"] for item in data]
    })


# Training dataset
train_ds = to_hf_dataset(train_data)

# Testing dataset
test_ds = to_hf_dataset(test_data)

print("Training dataset:")
print(train_ds)

print("\nTesting dataset:")
print(test_ds)

Training dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 136
})

Testing dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 34
})


In [11]:
train_ds = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

test_ds = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

print("Training dataset tokenized!")
print("Testing dataset tokenized!")

Map:   0%|          | 0/136 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Training dataset tokenized!
Testing dataset tokenized!


In [12]:
from transformers import (
    TrainingArguments,
    DataCollatorForTokenClassification
)

# Data ko batch mein properly arrange karne ke liye
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",

    # PDF ke according
    num_train_epochs=5,

    # Batch size
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Har epoch ke baad evaluation
    eval_strategy="epoch",

    # Har epoch ke baad model save
    save_strategy="epoch",

    # Har 10 steps ke baad log
    logging_steps=10,

    # Best model ko end mein use karo
    load_best_model_at_end=True,

    # Extra reporting band
    report_to="none",

    # GPU available ho to faster training
    fp16=torch.cuda.is_available()
)

print("Training settings ready!")

Training settings ready!


In [13]:
import torch

from transformers import (
    TrainingArguments,
    DataCollatorForTokenClassification
)

# Data ko batches mein properly arrange karne ke liye
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",

    # 5 epochs
    num_train_epochs=5,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Har epoch ke baad evaluation
    eval_strategy="epoch",

    # Har epoch ke baad model save
    save_strategy="epoch",

    # Har 10 steps ke baad logging
    logging_steps=10,

    # Best model ko training ke end par load karna
    load_best_model_at_end=True,

    # External reporting band
    report_to="none",

    # GPU available ho to mixed precision
    fp16=torch.cuda.is_available()
)

print("Training settings ready!")
print("GPU available:", torch.cuda.is_available())

Training settings ready!
GPU available: True


In [14]:
from transformers import Trainer
import transformers

print("Transformers version:", transformers.__version__)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
)

print("Trainer created successfully!")

Transformers version: 5.15.1
Trainer created successfully!


In [15]:
print("Model:", type(trainer.model).__name__)
print("Train samples:", len(trainer.train_dataset))
print("Test samples:", len(trainer.eval_dataset))
print("Trainer is ready!")

Model: XLMRobertaForTokenClassification
Train samples: 136
Test samples: 34
Trainer is ready!


In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.410048
2,0.568633,0.334829
3,0.342798,0.260397
4,0.231123,0.220269
5,0.174955,0.216500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=45, training_loss=0.31267125474082097, metrics={'train_runtime': 182.6827, 'train_samples_per_second': 3.722, 'train_steps_per_second': 0.246, 'total_flos': 7414295472384.0, 'train_loss': 0.31267125474082097, 'epoch': 5.0})

In [17]:
results = trainer.evaluate()

print("Evaluation Results:")
print(results)

Training Loss,Validation Loss,Epoch
0.174955,0.216500,5


Evaluation Results:
{'eval_loss': 0.21649956703186035}


In [18]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = trainer.predict(test_ds)

preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Padding / ignored labels (-100) ko remove karo
mask = labels != -100

accuracy = accuracy_score(
    labels[mask],
    preds[mask]
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 0.9078
Accuracy: 90.78%


In [19]:
save_path = "./xlm_roberta_token_classifier"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")
print("Saved at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
Saved at: ./xlm_roberta_token_classifier
